# Trabalho 2 — Processos Estocásticos — Resolução

Detecção e estimação de pulsos impulsivos imersos em ruído:

\[
y[n] = A\, s[n - n_0] + r[n]
\]

- Janela: 256 amostras, \(f_s = 1000\,\mathrm{Hz}\)
- Template \(s[n]\): 64 amostras

**Regra:** ajuste e limiar apenas no treino; o teste só é aberto no Passo 8.


## Passo 0 — Preparação e carregamento dos dados


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy import signal
from scipy.linalg import toeplitz, inv

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (11, 4)

FS = 1000.0
N_WIN = 256
N_TMPL = 64

PROJECT = Path.cwd()
candidates = [PROJECT / "dados", PROJECT.parent / "dados", PROJECT, PROJECT.parent]
DATA_DIR = next(p for p in candidates if (p / "template_pulso.csv").exists())
print("DATA_DIR:", DATA_DIR)


def load_template():
    df = pd.read_csv(DATA_DIR / "template_pulso.csv")
    s = df["s"].to_numpy(dtype=float)
    assert len(s) == N_TMPL
    return s


def load_windows(path):
    df = pd.read_csv(path)
    ids = np.sort(df["record_id"].unique())
    Y = np.stack([df.loc[df["record_id"] == i, "y"].to_numpy(dtype=float) for i in ids])
    assert Y.shape[1] == N_WIN
    return Y


def load_labels(path):
    return pd.read_csv(path).sort_values("record_id").reset_index(drop=True)


s = load_template()
Y_noise = load_windows(DATA_DIR / "ruido_treino.csv")
Y_train = load_windows(DATA_DIR / "sinais_treino.csv")
Y_test = load_windows(DATA_DIR / "sinais_teste.csv")
lab_train = load_labels(DATA_DIR / "rotulos_treino.csv")
lab_test = load_labels(DATA_DIR / "rotulos_teste.csv")

assert Y_noise.shape == (300, N_WIN)
assert Y_train.shape == (500, N_WIN)
assert Y_test.shape == (250, N_WIN)
assert len(lab_train) == 500 and len(lab_test) == 250
print("Shapes OK:", Y_noise.shape, Y_train.shape, Y_test.shape)
print("Eventos treino:", int(lab_train.event_present.sum()), "/", len(lab_train))
print("Eventos teste:", int(lab_test.event_present.sum()), "/", len(lab_test))


In [ ]:
def true_pulse(s, n0, A, n_win=N_WIN):
    y = np.zeros(n_win)
    if A == 0 or n0 < 0:
        return y
    end = min(n_win, n0 + len(s))
    y[n0:end] = A * s[: end - n0]
    return y


idx_pos = int(lab_train.index[lab_train.event_present == 1][0])
idx_neg = int(lab_train.index[lab_train.event_present == 0][0])
t0p = int(lab_train.loc[idx_pos, "t0_sample"])
Ap = float(lab_train.loc[idx_pos, "amplitude_A"])

fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
axes[0].plot(Y_train[idx_pos], lw=0.8)
axes[0].axvline(t0p, color="r", ls="--", label=f"n0={t0p}, A={Ap:.2f}")
axes[0].plot(true_pulse(s, t0p, Ap), lw=1.2, label="pulso verdadeiro")
axes[0].legend(); axes[0].set_title("Janela com pulso (treino)")
axes[1].plot(Y_train[idx_neg], lw=0.8, color="C1")
axes[1].set_title("Janela sem pulso (treino)"); axes[1].set_xlabel("n")
plt.tight_layout(); plt.show()

print("CHECKPOINT 0: dados carregados e exemplos alinhados com os rótulos.")


## Passo 1 — Análise do ruído


In [ ]:
def biased_autocorr(x, max_lag):
    x = np.asarray(x, dtype=float) - np.mean(x)
    n = len(x)
    full = np.correlate(x, x, mode="full") / n
    mid = n - 1
    return full[mid : mid + max_lag + 1]


MAX_LAG = 40
noise_flat = Y_noise.ravel()
mu_r = float(noise_flat.mean())
var_r = float(noise_flat.var(ddof=0))
std_r = float(np.sqrt(var_r))

R_mean = np.mean([biased_autocorr(Y_noise[i], MAX_LAG) for i in range(len(Y_noise))], axis=0)
R_norm = R_mean / R_mean[0]
band = 2 / np.sqrt(N_WIN)
outside = float(np.mean(np.abs(R_norm[1:]) > band))

f_psd, psd_r = signal.welch(noise_flat, fs=FS, nperseg=128, noverlap=64)
psd_ratio = float(psd_r.max() / psd_r.min())
noise_is_colored = outside > 0.05 or psd_ratio > 3.0

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].stem(np.arange(MAX_LAG + 1), R_norm, basefmt=" ")
axes[0].axhline(band, color="r", ls="--", lw=0.8)
axes[0].axhline(-band, color="r", ls="--", lw=0.8)
axes[0].set_title("Autocorrelação do ruído"); axes[0].set_xlabel("lag")
axes[1].semilogy(f_psd, psd_r)
axes[1].set_title("PSD do ruído (Welch)"); axes[1].set_xlabel("Hz")
plt.tight_layout(); plt.show()

print(f"média={mu_r:.4f}, var={var_r:.4f}, std={std_r:.4f}")
print(f"% lags fora da faixa branca: {100*outside:.1f}% | razão PSD max/min: {psd_ratio:.2f}")
print(f"Ruído colorido? {noise_is_colored}")

R_full = biased_autocorr(noise_flat - mu_r, N_WIN - 1)
C = toeplitz(R_full)
C = C + (1e-6 * np.trace(C) / N_WIN) * np.eye(N_WIN)
Cinv = inv(C)

print("CHECKPOINT 1: estatísticas do ruído e covariância estimadas.")


## Passo 2 — Análise do template


In [ ]:
NFFT = 256
t_s = np.arange(N_TMPL) / FS
S_f = np.fft.rfft(s, n=NFFT)
f_s = np.fft.rfftfreq(NFFT, d=1 / FS)
S_pow = np.abs(S_f) ** 2
f_peak = float(f_s[np.argmax(S_pow)])
energy_low = float(S_pow[f_s <= 100].sum() / S_pow.sum())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(t_s * 1000, s)
axes[0].set_xlabel("ms"); axes[0].set_title("Template s[n]")
psd_interp = np.interp(f_s, f_psd, psd_r)
axes[1].semilogy(f_s, S_pow / S_pow.max(), label="|S|² norm.")
axes[1].semilogy(f_s, psd_interp / psd_interp.max(), label="PSD ruído norm.", alpha=0.8)
axes[1].set_xlabel("Hz"); axes[1].legend(); axes[1].set_title("Pulso vs ruído")
plt.tight_layout(); plt.show()

print(f"Pico espectral do pulso ~ {f_peak:.1f} Hz; energia em 0–100 Hz: {100*energy_low:.1f}%")
print("CHECKPOINT 2: template caracterizado e comparado ao espectro do ruído.")


## Passo 3 — Filtro de Wiener


In [ ]:
def place_template(s, n0, n_win=N_WIN):
    v = np.zeros(n_win)
    end = min(n_win, n0 + len(s))
    if 0 <= n0 < n_win:
        v[n0:end] = s[: end - n0]
    return v


def apply_wiener_freq(y, H):
    Y = np.fft.rfft(y, n=NFFT)
    return np.fft.irfft(H * Y, n=NFFT)[: len(y)]


A_typ = float(lab_train.loc[lab_train.event_present == 1, "amplitude_A"].mean() ** 2)
s_mid = place_template(s, (N_WIN - N_TMPL) // 2)
Sxx = np.abs(np.fft.rfft(s_mid, n=NFFT)) ** 2 * A_typ
f_fft = np.fft.rfftfreq(NFFT, d=1 / FS)
Nxx = np.interp(f_fft, f_psd, psd_r)
H_w = Sxx / (Sxx + Nxx + 1e-18)

Y_train_w = np.stack([apply_wiener_freq(y, H_w) for y in Y_train])
Y_noise_w = np.stack([apply_wiener_freq(y, H_w) for y in Y_noise])
Y_test_w = np.stack([apply_wiener_freq(y, H_w) for y in Y_test])

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
axes[0, 0].plot(Y_train[idx_pos], lw=0.8); axes[0, 0].axvline(t0p, color="r", ls="--")
axes[0, 0].set_title("Com pulso — bruto")
axes[0, 1].plot(Y_train_w[idx_pos], lw=0.8, color="C2"); axes[0, 1].axvline(t0p, color="r", ls="--")
axes[0, 1].set_title("Com pulso — Wiener")
axes[1, 0].plot(Y_train[idx_neg], lw=0.8); axes[1, 0].set_title("Só ruído — bruto")
axes[1, 1].plot(Y_train_w[idx_neg], lw=0.8, color="C2"); axes[1, 1].set_title("Só ruído — Wiener")
plt.tight_layout(); plt.show()

# RMSE forma: Wiener direto vs bruto
def rmse_vs_true(Y_est, labels):
    errs = []
    for i, row in labels.iterrows():
        if int(row.event_present) != 1:
            continue
        true = true_pulse(s, int(row.t0_sample), float(row.amplitude_A))
        errs.append(np.sqrt(np.mean((Y_est[i] - true) ** 2)))
    return float(np.mean(errs))

print(f"RMSE forma (bruto):  {rmse_vs_true(Y_train, lab_train):.3f}")
print(f"RMSE forma (Wiener): {rmse_vs_true(Y_train_w, lab_train):.3f}")
print("CHECKPOINT 3: Wiener projetado e comparado no treino (teste ainda fechado).")


## Passo 4 — Filtro casado


In [ ]:
def matched_filter(y, s):
    out = signal.correlate(y, s, mode="valid")
    k = int(np.argmax(out))
    return out, k, float(out[k])


def blue_amplitude(y, s_aligned, Cinv=None):
    if np.allclose(s_aligned, 0):
        return 0.0
    if Cinv is None:
        num = float(np.dot(s_aligned, y))
        den = float(np.dot(s_aligned, s_aligned))
    else:
        Cs = Cinv @ s_aligned
        num = float(np.dot(Cs, y))
        den = float(np.dot(Cs, s_aligned))
    return num / den if den > 0 else 0.0


scores_c, n0_c = [], []
for y in Y_train:
    _, n0, sc = matched_filter(y, s)
    scores_c.append(sc); n0_c.append(n0)
scores_c = np.asarray(scores_c); n0_c = np.asarray(n0_c)

mask_pos = lab_train.event_present.to_numpy() == 1
mask_neg = ~mask_pos
err_n0 = np.abs(n0_c[mask_pos] - lab_train.t0_sample.to_numpy()[mask_pos])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(scores_c[mask_neg], bins=30, alpha=0.7, density=True, label="sem pulso")
axes[0].hist(scores_c[mask_pos], bins=30, alpha=0.7, density=True, label="com pulso")
axes[0].legend(); axes[0].set_title("Estatística do casado (treino)")
axes[1].hist(err_n0, bins=30, color="C1")
axes[1].set_title("Erro |n0_hat - n0|"); axes[1].set_xlabel("amostras")
plt.tight_layout(); plt.show()

print(f"Erro n0 — média={err_n0.mean():.2f}, mediana={np.median(err_n0):.2f}")

# Wiener + casado
scores_wc, n0_wc = [], []
for y in Y_train_w:
    _, n0, sc = matched_filter(y, s)
    scores_wc.append(sc); n0_wc.append(n0)
scores_wc = np.asarray(scores_wc); n0_wc = np.asarray(n0_wc)
err_n0_wc = np.abs(n0_wc[mask_pos] - lab_train.t0_sample.to_numpy()[mask_pos])
print(f"Wiener+casado — mediana |Δn0|={np.median(err_n0_wc):.2f}")
print("CHECKPOINT 4: casado e Wiener+casado com separação de classes no treino.")


## Passo 5 — Estimação de amplitude (BLUE)


In [ ]:
A_true = lab_train.amplitude_A.to_numpy()
A_blue_w = np.zeros(len(Y_train))
A_blue_c = np.zeros(len(Y_train))
for i in range(len(Y_train)):
    sal = place_template(s, int(n0_c[i]))
    A_blue_w[i] = blue_amplitude(Y_train[i], sal, None)
    A_blue_c[i] = blue_amplitude(Y_train[i], sal, Cinv)

A_naive = np.array([float(np.max(y)) for y in Y_train])

def rmse_A(Ahat, mask=None):
    m = mask_pos if mask is None else mask
    return float(np.sqrt(np.mean((Ahat[m] - A_true[m]) ** 2)))

print(f"RMSE A naive (max y):     {rmse_A(A_naive):.3f}")
print(f"RMSE A BLUE branco:       {rmse_A(A_blue_w):.3f}")
print(f"RMSE A BLUE colorido:     {rmse_A(A_blue_c):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, Ah, title in zip(axes, [A_blue_w, A_blue_c], ["BLUE branco", "BLUE colorido"]):
    ax.scatter(A_true[mask_pos], Ah[mask_pos], s=12, alpha=0.6)
    lim = [0, max(A_true[mask_pos].max(), Ah[mask_pos].max()) * 1.05]
    ax.plot(lim, lim, "r--")
    ax.set_xlabel("A verdadeira"); ax.set_ylabel("A estimada"); ax.set_title(title)
plt.tight_layout(); plt.show()
print("CHECKPOINT 5: BLUE melhor que baseline ingênuo no treino.")


## Passo 6 — Limiar de detecção (FA ≈ 1%)


In [ ]:
scores_noise = np.array([matched_filter(y, s)[2] for y in Y_noise])
scores_noise_wc = np.array([matched_filter(y, s)[2] for y in Y_noise_w])
scores_noise_raw = np.array([float(np.max(np.abs(y))) for y in Y_noise])
scores_noise_wraw = np.array([float(np.max(np.abs(y))) for y in Y_noise_w])

thr_casado = float(np.quantile(scores_noise, 0.99))
thr_wc = float(np.quantile(scores_noise_wc, 0.99))
thr_raw = float(np.quantile(scores_noise_raw, 0.99))
thr_wraw = float(np.quantile(scores_noise_wraw, 0.99))

fa_emp = float(np.mean(scores_noise >= thr_casado))
print(f"Limiar casado (P99) = {thr_casado:.4f} | FA empírica no ruído = {100*fa_emp:.2f}%")
print(f"Limiar Wiener+casado = {thr_wc:.4f}")
print(f"Limiar bruto (máx|y|) = {thr_raw:.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(scores_noise, bins=40, alpha=0.75, label="ruído puro")
ax.axvline(thr_casado, color="r", ls="--", label=f"limiar={thr_casado:.3f}")
ax.legend(); ax.set_title("Definição do limiar (H0)")
plt.tight_layout(); plt.show()

print("CHECKPOINT 6: limiares congelados a partir do ruído de treino.")


## Passo 7 — Avaliação completa no treino

Métricas: ROC/AUC, matriz de confusão (com limiar), RMSE da forma, RMSE de amplitude e erro de \(n_0\).


In [ ]:
def roc_curve_scores(y_true, scores):
    order = np.argsort(-scores)
    y_true = np.asarray(y_true)[order]
    scores = np.asarray(scores)[order]
    P = np.sum(y_true == 1); N = np.sum(y_true == 0)
    tpr, fpr = [0.0], [0.0]
    tp = fp = 0
    prev = None
    for yt, sc in zip(y_true, scores):
        if prev is None or sc != prev:
            tpr.append(tp / P if P else 0.0)
            fpr.append(fp / N if N else 0.0)
            prev = sc
        if yt == 1:
            tp += 1
        else:
            fp += 1
    tpr += [tp / P if P else 0.0, 1.0]
    fpr += [fp / N if N else 0.0, 1.0]
    return np.array(fpr), np.array(tpr)


def auc_trapz(fpr, tpr):
    order = np.argsort(fpr)
    return float(np.trapezoid(tpr[order], fpr[order]))


def detection_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return dict(tp=tp, tn=tn, fp=fp, fn=fn, precision=prec, recall=rec, f1=f1)


def eval_set(Y, labels, Yw):
    y_true = labels.event_present.to_numpy().astype(int)
    configs = {
        "bruto": (Y, "raw", thr_raw, False),
        "wiener": (Yw, "raw", thr_wraw, False),
        "casado": (Y, "matched", thr_casado, False),
        "wiener_casado": (Yw, "matched", thr_wc, False),
        "blue_branco": (Y, "matched", thr_casado, False),
        "blue_colorido": (Y, "matched", thr_casado, True),
    }
    results = {}
    for name, (Y_in, mode, thr, colored) in configs.items():
        scores = np.zeros(len(Y_in))
        n0h = np.zeros(len(Y_in), dtype=int)
        Ah = np.zeros(len(Y_in))
        Yh = np.zeros_like(Y)
        for i in range(len(Y_in)):
            _, n0, sc = matched_filter(Y_in[i], s)
            n0h[i] = n0
            scores[i] = sc if mode == "matched" else float(np.max(np.abs(Y_in[i])))
            sal = place_template(s, n0)
            Ah[i] = blue_amplitude(Y[i], sal, Cinv if colored else None)
            Yh[i] = Ah[i] * sal
        y_pred = (scores >= thr).astype(int)
        fpr, tpr = roc_curve_scores(y_true, scores)
        # waveform RMSE
        if name in ("bruto", "wiener"):
            rmse_wave = rmse_vs_true(Y_in, labels)
        else:
            errs = []
            for i, row in labels.iterrows():
                if int(row.event_present) != 1:
                    continue
                true = true_pulse(s, int(row.t0_sample), float(row.amplitude_A))
                errs.append(np.sqrt(np.mean((Yh[i] - true) ** 2)))
            rmse_wave = float(np.mean(errs))
        mpos = y_true == 1
        rmse_amp = float(np.sqrt(np.mean((Ah[mpos] - labels.amplitude_A.to_numpy()[mpos]) ** 2)))
        n0_err = float(np.mean(np.abs(n0h[mpos] - labels.t0_sample.to_numpy()[mpos])))
        n0_med = float(np.median(np.abs(n0h[mpos] - labels.t0_sample.to_numpy()[mpos])))
        results[name] = dict(
            auc=auc_trapz(fpr, tpr), fpr=fpr, tpr=tpr, det=detection_metrics(y_true, y_pred),
            rmse_wave=rmse_wave, rmse_amp=rmse_amp, n0_err=n0_err, n0_med=n0_med,
            scores=scores, A_hat=Ah, y_hat=Yh, n0_hat=n0h, y_pred=y_pred,
        )
    return results


train_eval = eval_set(Y_train, lab_train, Y_train_w)

rows = []
for name, r in train_eval.items():
    d = r["det"]
    rows.append({
        "método": name, "AUC": r["auc"], "F1": d["f1"], "prec": d["precision"], "rec": d["recall"],
        "RMSE_forma": r["rmse_wave"], "RMSE_A": r["rmse_amp"],
        "n0_mean": r["n0_err"], "n0_med": r["n0_med"],
        "TP": d["tp"], "FP": d["fp"], "FN": d["fn"], "TN": d["tn"],
    })
df_train = pd.DataFrame(rows)
display(df_train.round(3))

fig, ax = plt.subplots(figsize=(6.5, 5))
for name, label in [("bruto", "Bruto"), ("wiener", "Wiener"), ("casado", "Casado"), ("wiener_casado", "Wiener+casado")]:
    r = train_eval[name]
    ax.plot(r["fpr"], r["tpr"], label=f"{label} AUC={r['auc']:.3f}")
ax.plot([0, 1], [0, 1], "k--", lw=0.8)
ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.set_title("ROC — treino"); ax.legend()
plt.tight_layout(); plt.show()

best_wave = min(train_eval, key=lambda k: train_eval[k]["rmse_wave"])
best_det = max(["bruto", "wiener", "casado", "wiener_casado"], key=lambda k: train_eval[k]["auc"])
best_amp = min(["blue_branco", "blue_colorido", "casado"], key=lambda k: train_eval[k]["rmse_amp"])
print(f"Escolha (treino): forma={best_wave}, detecção={best_det}, amplitude={best_amp}")
print("CHECKPOINT 7: métricas de treino congeladas. Abrindo o teste a seguir.")


## Passo 8 — Avaliação no teste (uma vez)

Parâmetros e limiares **congelados**. Sem retreino.


In [ ]:
test_eval = eval_set(Y_test, lab_test, Y_test_w)

rows = []
for name, r in test_eval.items():
    d = r["det"]
    rows.append({
        "método": name, "AUC": r["auc"], "F1": d["f1"], "prec": d["precision"], "rec": d["recall"],
        "RMSE_forma": r["rmse_wave"], "RMSE_A": r["rmse_amp"],
        "n0_mean": r["n0_err"], "n0_med": r["n0_med"],
        "TP": d["tp"], "FP": d["fp"], "FN": d["fn"], "TN": d["tn"],
    })
df_test = pd.DataFrame(rows)
display(df_test.round(3))

fig, ax = plt.subplots(figsize=(6.5, 5))
for name, label in [("bruto", "Bruto"), ("wiener", "Wiener"), ("casado", "Casado"), ("wiener_casado", "Wiener+casado")]:
    r = test_eval[name]
    ax.plot(r["fpr"], r["tpr"], label=f"{label} AUC={r['auc']:.3f}")
ax.plot([0, 1], [0, 1], "k--", lw=0.8)
ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.set_title("ROC — teste"); ax.legend()
plt.tight_layout(); plt.show()

print("Comparação treino vs teste (casado):")
print(f"  AUC  {train_eval['casado']['auc']:.3f} → {test_eval['casado']['auc']:.3f}")
print(f"  F1   {train_eval['casado']['det']['f1']:.3f} → {test_eval['casado']['det']['f1']:.3f}")
print(f"  RMSE forma {train_eval['casado']['rmse_wave']:.3f} → {test_eval['casado']['rmse_wave']:.3f}")
print(f"  RMSE A {train_eval['blue_colorido']['rmse_amp']:.3f} → {test_eval['blue_colorido']['rmse_amp']:.3f}")
print("CHECKPOINT 8: avaliação final no teste concluída.")


## Conclusão rápida

- **Reconstrução da forma:** melhor com estimativa \(\hat{A}\,s[n-\hat{n}_0]\) (BLUE), sobretudo com covariância do ruído quando colorido.
- **Detecção:** filtro casado (e Wiener+casado) supera o limiar no máximo de \(|y|\); limiar a FA≈1% é conservador (muitos FN, F1 moderado).
- **Amplitude:** BLUE colorido tende a reduzir o RMSE em relação ao BLUE branco / baseline.
